# 1. 패키지 설치

In [1]:
# 패키지 설치
%pip install -U python-dotenv langchain langchain-openai langchain-community langchain-text-splitters docx2txt langchain-chroma langchain-classic

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


# 2. Knowledge Base 구성을 위한 데이터 생성

- `RecursiveCharacterTextSplitter`를 활용한 데이터 chunking
    - split 된 chunk를 LLM에 전달하면 토큰 절약 (비용·답변시간 감소)
- `chunk_size`: chunk 최대 크기 / `chunk_overlap`: 앞뒤 chunk가 겹치는 정도
- 02-markdown 과 동일하되 **표를 table 형식으로 변환한 docx**(`tax_with_table.docx`) 사용

In [3]:
# docx 문서 Load & Split (표를 table 형식으로 변환한 docx)
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200,
)

loader = Docx2txtLoader('./tax_with_table.docx')
document_list = loader.load_and_split(text_splitter=text_splitter)

In [ ]:
# ⚠️ docx '진짜 표(table)'는 Docx2txtLoader가 파이프(|) 없이 셀을 줄단위로 펼쳐 평문화함
#  → GPT가 표 구조(과세표준↔세율 대응)를 이해하기 어려움
#  → 강의도 table 먼저 썼다가 이 문제로 markdown(| ... |)으로 전환 (markdown이 LLM 이해에 유리)
document_list

In [5]:
# 환경변수 Load + Embedding 설정
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings

load_dotenv()

embedding = OpenAIEmbeddings(model='text-embedding-3-large')

In [7]:
# Vector DB(Chroma)
from langchain_chroma import Chroma

# 데이터 처음 저장
# database = Chroma.from_documents(documents=document_list, embedding=embedding, collection_name='chroma-tax', persist_directory='./chroma_table')

# 이미 저장된 데이터 사용
database = Chroma(collection_name='chroma-tax', persist_directory='./chroma_table', embedding_function=embedding)

# 3. 답변 생성을 위한 Retrieval

- `create_retrieval_chain`은 retriever 객체를 입력으로 받음 → `as_retriever()` 사용

In [8]:
query = '연봉 5천만원인 직장인의 소득세는 얼마인가요?'

# create_retrieval_chain용 retriever 생성 (k=10)
retriever = database.as_retriever(search_kwargs={"k": 10})

In [ ]:
# retriever 동작 확인
# ⚠️ 세율표(제55조)가 7번째 — "직장인"(질문) vs "거주자"(표) 용어 불일치로 유사도 낮음 (→ 3.6 키워드 사전)
# 검색 순위는 markdown본과 비슷, 차이는 LLM이 chunk를 이해하는 단계(평탄화 table < markdown |)
retriever.invoke(query)

# 4. Augmentation을 위한 Prompt 활용

- `create_retrieval_chain` 전용 프롬프트(`langchain-ai/retrieval-qa-chat`, 변수 input+context)를 Hub에서 가져옴

In [10]:
# LangChain 기반 LLM 설정
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model='gpt-4o-mini')

In [11]:
# 프롬프트: create_retrieval_chain용 retrieval-qa-chat (System + chat_history + Human, 변수 input·context)
# langchain.hub 제거, langchain_classic.hub.pull도 deprecated → langsmith로 pull
from langsmith import Client

client = Client()
retrieval_qa_chat_prompt = client.pull_prompt('langchain-ai/retrieval-qa-chat', dangerously_pull_public_prompt=True)

# 5. 답변 생성

- `create_stuff_documents_chain`(문서 결합) + `create_retrieval_chain`(검색→생성)으로 RAG 체인 구성
    - `RetrievalQA`(3.2)의 대체. v1에서 `langchain_classic`에 위치하나 deprecated 아님

In [12]:
# v1: create_stuff_documents_chain·create_retrieval_chain 은 langchain_classic 으로 이동 (deprecated 아님)
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain

combine_docs_chain = create_stuff_documents_chain(llm, retrieval_qa_chat_prompt)
retrieval_chain = create_retrieval_chain(retriever, combine_docs_chain)

In [13]:
# Chain을 통한 LLM 호출
ai_msg = retrieval_chain.invoke({"input": query})

In [14]:
# 출력 확인
ai_msg

{'input': '연봉 5천만원인 직장인의 소득세는 얼마인가요?',
 'context': [Document(id='b773afb5-acf5-4808-89c7-c500bdaa5558', metadata={'source': './tax_with_table.docx'}, page_content='나. 그 밖의 배당소득에 대해서는 100분의 14\n\n3. 원천징수대상 사업소득에 대해서는 100분의 3. 다만, 외국인 직업운동가가 한국표준산업분류에 따른 스포츠 클럽 운영업 중 프로스포츠구단과의 계약(계약기간이 3년 이하인 경우로 한정한다)에 따라 용역을 제공하고 받는 소득에 대해서는 100분의 20으로 한다.\n\n4. 근로소득에 대해서는 기본세율. 다만, 일용근로자의 근로소득에 대해서는 100분의 6으로 한다.\n\n5. 공적연금소득에 대해서는 기본세율\n\n5의2.제20조의3제1항제2호나목 및 다목에 따른 연금계좌 납입액이나 운용실적에 따라 증가된 금액을 연금수령한 연금소득에 대해서는 다음 각 목의 구분에 따른 세율. 이 경우 각 목의 요건을 동시에 충족하는 때에는 낮은 세율을 적용한다.\n\n가. 연금소득자의 나이에 따른 다음의 세율\n\n\n\n나. 삭제<2014. 12. 23.>\n\n다. 사망할 때까지 연금수령하는 대통령령으로 정하는 종신계약에 따라 받는 연금소득에 대해서는 100분의 4\n\n5의3. 제20조의3제1항제2호가목에 따라 퇴직소득을 연금수령하는 연금소득에 대해서는 다음 각 목의 구분에 따른 세율. 이 경우 연금 실제 수령연차 및 연금외수령 원천징수세율의 구체적인 내용은 대통령령으로 정한다.\n\n가. 연금 실제 수령연차가 10년 이하인 경우: 연금외수령 원천징수세율의 100분의 70\n\n나. 연금 실제 수령연차가 10년을 초과하는 경우: 연금외수령 원천징수세율의 100분의 60\n\n6. 기타소득에 대해서는 다음에 규정하는 세율. 다만, 제8호를 적용받는 경우는 제외한다.\n\n가. 제14조제3항제8호라목 및 마목에 해당하는 소득금액이 3